In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt 
import os 
con =  duckdb.connect(database=':memory:')

con.execute ( """
    create table dim_customers as 
     select * from read_parquet('dim_customers.parquet') ; 

    create table fact_calls as 
    select * from read_parquet('fact_calls.parquet') ; 

"""
            )



export_folder = "powerbi_dashboard_files" 

if not os.path.exists(export_folder):
    os.makedirs(export_folder)
    print(f"Created new directory: '{export_folder}'")
else:
    print(f"Directory '{export_folder}' already exists.")


con.execute(f"""
    COPY (
   with raw_data AS ( 
    SELECT 
        call_id , 
        customer_id , 
        timestamp , 
        date_trunc('month' , timestamp) as billing_month , 
        hour(timestamp) as call_hour , 
        direction , 
        caller_id_country , 
        destination_country , 
        duration_minutes , 
        rate_price , 
        rate_cost , 
        -- we need to make sure that we handle the 2000 free inbound minutes 
        sum(case when direction = 'inbound' then duration_minutes else 0 end)
            over (partition by customer_id , billing_month order by timestamp 
            rows between unbounded PRECEDING and current row  ) as running_inbound_total , 
        rate_cost * duration_minutes as call_cost  , 
        case 
        when  direction = 'outbound' then (duration_minutes * rate_price)
        when  direction = 'inbound'  then 
        case 
        when running_inbound_total <= 2000 then 0.0
        when  (running_inbound_total - duration_minutes) < 2000 AND running_inbound_total > 2000  then (running_inbound_total - 2000) * 0.02
        else duration_minutes * 0.02 end end as call_revenue , 
        sip_code , 
                
    FROM fact_calls 
    )                      
    select 
    date(timestamp) as date , 
    call_hour  , 
    direction , 
    count(distinct call_id) as total_calls  , 
    sum(duration_minutes) as total_duration_minutes , 
    sum(call_revenue) as total_revenues , 
    sum(call_cost) as total_carrier_cost , 
    count(distinct if(sip_code >= 500 , call_id , null)) as sip_5xx_errors
    from raw_data
    group by all 
        
    ) TO '{export_folder}/pbi_daily_health.parquet' (FORMAT PARQUET);
""")
print(" pbi_daily_health.parquet saved!")




con.execute(f"""
    COPY (
        with raw_data AS ( 
    SELECT 
        call_id , 
        customer_id , 
        timestamp , 
        date_trunc('month' , timestamp) as billing_month , 
        hour(timestamp) as call_hour , 
        direction , 
        caller_id_country , 
        destination_country , 
        duration_minutes , 
        rate_price , 
        rate_cost , 
        sum(case when direction = 'inbound' then duration_minutes else 0 end)
            over (partition by customer_id , billing_month order by timestamp 
            rows between unbounded PRECEDING and current row  ) as running_inbound_total , 
        rate_cost * duration_minutes as call_cost  , 
        case 
        when  direction = 'outbound' then (duration_minutes * rate_price)
        when  direction = 'inbound'  then 
        case 
        when running_inbound_total <= 2000 then 0.0
        when  (running_inbound_total - duration_minutes) < 2000 AND running_inbound_total > 2000  then (running_inbound_total - 2000) * 0.02
        else duration_minutes * 0.02 end end as call_revenue , 
        sip_code , 
                
    FROM fact_calls 
      )
    select
    date(timestamp) as date , 
    customer_id ,
    destination_country , 
    direction , 
    count(distinct call_id) as total_calls  , 
    sum(duration_minutes) as total_duration_minutes , 
    sum(call_revenue) as total_revenues , 
    sum(call_cost) as total_carrier_cost , 
    total_revenues - total_carrier_cost as net_margin , 
    (total_revenues - total_carrier_cost) / nullif(total_revenues, 0) as margin_percentage
    from raw_data
    group by all 
    

    ) TO '{export_folder}/pbi_route_margins.parquet' (FORMAT PARQUET);
""")
print("pbi_route_margins.parquet saved!")

con.execute(f"""
            
copy (


with monthly_usage as (
select
customer_id , 
date_trunc('month',timestamp) as billing_month , 
count(distinct agent_id) as active_agents  , 
count(distinct call_id) as total_calls  , 
max(date(timestamp)) as last_call_date 
from fact_calls 
group by 1, 2
) 
                                  
select 
u.billing_month , 
c.customer_id , 
c.company_name , 
c.paid_seats  , 
(c.paid_seats * c.seat_price) as monthly_seat_revenues , 
coalesce(u.active_agents , 0) active_agents , 
                                  
case 
    when c.paid_seats = 0 OR c.paid_seats is null then null 
    else (u.active_agents * 1.0) / c.paid_seats 
    end AS utilization_percentage,

coalesce(u.total_calls,0) as total_call_volume , 
                                  
u.last_call_date 
from dim_customers c 
left join monthly_usage u on c.customer_id = u.customer_id 

        
    ) TO '{export_folder}/pbi_customer_health.parquet' (FORMAT PARQUET);
""")
print(" pbi_customer_health.parquet saved!")


Directory 'powerbi_dashboard_files' already exists.
 pbi_daily_health.parquet saved!
pbi_route_margins.parquet saved!
 pbi_customer_health.parquet saved!
